## Keycloak: 모든 유저의 sn / givenName 조회

이 노트북은 Keycloak Admin REST API를 이용해 **모든 유저의 `sn`, `givenName`** 를 수집하고,
`sn`에 공백이 포함되는 등 **풀네임이 들어갔을 가능성이 있는 케이스**를 따로 추려 CSV로 저장합니다.

- **변경(수정/업데이트)은 하지 않습니다.** (조회 + 리포트만)
- 인증정보는 **환경변수 또는 실행 시 입력(getpass)** 으로 받습니다.

### 필요한 정보
- `KEYCLOAK_BASE_URL` 예: `http://192.168.2.59:8080`
- `KEYCLOAK_REALM` 예: `sso`
- `KEYCLOAK_CLIENT_ID` 예: `admin-cli`
- `KEYCLOAK_USERNAME`
- `KEYCLOAK_PASSWORD`

> 토큰: `BASE_URL/realms/{REALM}/protocol/openid-connect/token`
> 유저목록: `BASE_URL/admin/realms/{REALM}/users`


In [ ]:
# 필요하면 설치 (이미 설치돼있으면 스킵)
%pip -q install requests pandas


In [ ]:
import os
from getpass import getpass
from typing import Any, Dict, List, Optional, Tuple

import requests
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)


In [ ]:
def _env_or_prompt(name: str, *, secret: bool = False, default: Optional[str] = None) -> str:
    v = os.getenv(name)
    if v:
        return v
    if default is not None and not secret:
        prompt = f"{name} [{default}]: "
        inp = input(prompt).strip()
        return inp or default
    if secret:
        return getpass(f"{name}: ")
    return input(f"{name}: ").strip()


def get_admin_token(
    base_url: str,
    realm: str,
    client_id: str,
    username: str,
    password: str,
    *,
    verify_tls: bool = True,
    timeout: int = 30,
) -> str:
    token_url = f"{base_url.rstrip('/')}/realms/{realm}/protocol/openid-connect/token"
    data = {
        "grant_type": "password",
        "client_id": client_id,
        "username": username,
        "password": password,
    }
    r = requests.post(token_url, data=data, timeout=timeout, verify=verify_tls)
    r.raise_for_status()
    j = r.json()
    if "access_token" not in j:
        raise RuntimeError(f"No access_token in response: {j}")
    return j["access_token"]


def fetch_all_users(
    base_url: str,
    realm: str,
    token: str,
    *,
    page_size: int = 100,
    verify_tls: bool = True,
    timeout: int = 30,
) -> List[Dict[str, Any]]:
    # Keycloak은 /users?first=0&max=100 같은 방식으로 페이징
    users_url = f"{base_url.rstrip('/')}/admin/realms/{realm}/users"
    headers = {"Authorization": f"Bearer {token}"}

    all_users: List[Dict[str, Any]] = []
    first = 0
    while True:
        params = {"first": first, "max": page_size}
        r = requests.get(users_url, headers=headers, params=params, timeout=timeout, verify=verify_tls)
        r.raise_for_status()
        batch = r.json()
        if not isinstance(batch, list):
            raise RuntimeError(f"Unexpected users response: {batch}")
        if not batch:
            break
        all_users.extend(batch)
        first += len(batch)
    return all_users


In [ ]:
# 접속 정보 (환경변수 우선, 없으면 입력)
KEYCLOAK_BASE_URL = _env_or_prompt("KEYCLOAK_BASE_URL", default="http://192.168.2.59:8080")
KEYCLOAK_REALM = _env_or_prompt("KEYCLOAK_REALM", default="sso")
KEYCLOAK_CLIENT_ID = _env_or_prompt("KEYCLOAK_CLIENT_ID", default="admin-cli")
KEYCLOAK_USERNAME = _env_or_prompt("KEYCLOAK_USERNAME")
KEYCLOAK_PASSWORD = _env_or_prompt("KEYCLOAK_PASSWORD", secret=True)

_verify_raw = os.getenv("KEYCLOAK_VERIFY_TLS", "true").strip().lower()
KEYCLOAK_VERIFY_TLS = _verify_raw not in {"0", "false", "no"}
KEYCLOAK_PAGE_SIZE = int(os.getenv("KEYCLOAK_PAGE_SIZE", "100"))

print("BASE_URL:", KEYCLOAK_BASE_URL)
print("REALM:", KEYCLOAK_REALM)
print("CLIENT_ID:", KEYCLOAK_CLIENT_ID)
print("USERNAME:", KEYCLOAK_USERNAME)
print("VERIFY_TLS:", KEYCLOAK_VERIFY_TLS)
print("PAGE_SIZE:", KEYCLOAK_PAGE_SIZE)


In [ ]:
token = get_admin_token(
    KEYCLOAK_BASE_URL,
    KEYCLOAK_REALM,
    KEYCLOAK_CLIENT_ID,
    KEYCLOAK_USERNAME,
    KEYCLOAK_PASSWORD,
    verify_tls=KEYCLOAK_VERIFY_TLS,
)

users = fetch_all_users(
    KEYCLOAK_BASE_URL,
    KEYCLOAK_REALM,
    token,
    page_size=KEYCLOAK_PAGE_SIZE,
    verify_tls=KEYCLOAK_VERIFY_TLS,
)

print("유저 수:", len(users))
# 샘플 구조 확인
users[0].keys() if users else None


In [ ]:
def _attr_first(user: Dict[str, Any], key: str) -> Optional[str]:
    """Keycloak user.attributes[key]가 list일 수도/str일 수도 있어서 첫 값을 안전하게 꺼냄."""
    attrs = user.get("attributes") or {}
    v = attrs.get(key)
    if v is None:
        return None
    if isinstance(v, list):
        return v[0] if v else None
    if isinstance(v, str):
        return v
    return str(v)


def normalize_name(s: Optional[str]) -> Optional[str]:
    if s is None:
        return None
    s = " ".join(str(s).strip().split())
    return s or None


def split_fullname_from_sn(sn: Optional[str]) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    """sn에 풀네임이 들어간 것 같을 때 후보 분리.

    반환: (candidate_sn, candidate_givenName, rule)
    - 'Last, First' 또는 공백 기반 토큰 분리만 제공합니다.
    """
    sn = normalize_name(sn)
    if not sn:
        return None, None, None

    # 'Last, First'
    if "," in sn:
        parts = [p.strip() for p in sn.split(",") if p.strip()]
        if len(parts) >= 2:
            cand_sn = parts[0]
            cand_gn = " ".join(parts[1:])
            return cand_sn or None, cand_gn or None, "comma"

    # 공백 기반
    toks = sn.split(" ")
    if len(toks) >= 2:
        cand_sn = toks[0]
        cand_gn = " ".join(toks[1:])
        return cand_sn or None, cand_gn or None, "space"

    return None, None, None


rows = []
for u in users:
    sn_attr = normalize_name(_attr_first(u, "sn"))
    gn_attr = normalize_name(_attr_first(u, "givenName"))

    # Keycloak 기본 필드도 같이 보이면 디버깅이 쉬움
    first_field = normalize_name(u.get("firstName"))
    last_field = normalize_name(u.get("lastName"))

    cand_sn, cand_gn, rule = split_fullname_from_sn(sn_attr)

    rows.append(
        {
            "id": u.get("id"),
            "username": u.get("username"),
            "email": u.get("email"),
            "enabled": u.get("enabled"),
            "sn": sn_attr,
            "givenName": gn_attr,
            "firstName(field)": first_field,
            "lastName(field)": last_field,
            "sn_has_space": bool(sn_attr and " " in sn_attr),
            "sn_has_comma": bool(sn_attr and "," in sn_attr),
            "givenName_missing": gn_attr is None,
            "candidate_sn": cand_sn,
            "candidate_givenName": cand_gn,
            "split_rule": rule,
        }
    )

df = pd.DataFrame(rows)
df.head(10)


In [ ]:
# 풀네임 의심 기준 (필요하면 여기 조건만 조정)
# - sn에 공백/콤마가 있다
# - 그리고 givenName이 비어있거나, givenName이 sn에 포함되는 이상 케이스
suspicious = df[
    (df["sn_has_space"] | df["sn_has_comma"]) & (
        df["givenName_missing"] | (
            df["sn"].notna() & df["givenName"].notna() & df.apply(lambda r: str(r["givenName"]) in str(r["sn"]), axis=1)
        )
    )
].copy()

print("의심 케이스 수:", len(suspicious))

# 저장
out_all = "keycloak_users_sn_givenName_all.csv"
out_susp = "keycloak_users_sn_givenName_suspicious.csv"

df.to_csv(out_all, index=False)
suspicious.to_csv(out_susp, index=False)

print("저장 완료:", out_all)
print("저장 완료:", out_susp)

# 화면에 일부 표시
suspicious[["username", "email", "sn", "givenName", "candidate_sn", "candidate_givenName", "split_rule"]].head(50)


### 실행 방법

1) 환경변수로 주입(추천)

```bash
export KEYCLOAK_BASE_URL='http://192.168.2.59:8080'
export KEYCLOAK_REALM='sso'
export KEYCLOAK_CLIENT_ID='admin-cli'
export KEYCLOAK_USERNAME='admin'
export KEYCLOAK_PASSWORD='***'
# (옵션) TLS 자체서명 등으로 검증 끄기
# export KEYCLOAK_VERIFY_TLS='false'
```

2) 노트북 실행 후 셀을 위에서부터 실행

### 결과물
- `keycloak_users_sn_givenName_all.csv`: 전체 유저의 sn/givenName 덤프
- `keycloak_users_sn_givenName_suspicious.csv`: `sn`에 풀네임이 들어갔을 가능성이 큰 유저 목록

> `candidate_sn` / `candidate_givenName` 는 **자동 수정값이 아니라 분리 후보**입니다. 실제 업데이트는 별도 작업으로 분리하는 걸 권장합니다.
